# silver_programs

Promotes the programs reference table from bronze to silver. Same shape as `silver_carriers`: small reference dimension, CSV-only, no UNION, no rejects, no business logic. Programs is the second of the two pass-through dimensions; after this, every remaining silver table involves reconciliation work.

## Why programs matters more than its row count suggests

Only 10 rows, but programs is the table that anchors the demo's domain story. Each program code (DIET, HCP, ENRG, CONT, etc.) is one of Citadel's actual specialty verticals, and the the 'commission_modifier' column on this table is one of the five rules driving the showcase commission calculation — we added it to the source generator after noticing the per-program multipliers were only hardcoded in the commission script and not stored as a queryable reference

## Source and output

Reads from `bronze_citadel_mga.bronze_programs` (10 rows, CSV-sourced).
Writes to `silver_citadel_mga.silver_programs`.

Lineage pattern is identical to silver_carriers:
- `_silver_load_timestamp`: when silver promoted this row
- `_silver_batch_id`: which silver run produced it
- `_bronze_batch_id`: carried forward from bronze for traceability

## Design choices

Same as silver_carriers. Schema enforcement on, explicit types, full rebuild on every run, idempotent.

## What's NOT here

No reject table. Programs is a controlled reference list maintained by underwriting; an invalid program code in source data would be a process failure to investigate, not a row to quietly quarantine. If that ever became a real concern, the right fix is upstream validation in IMS, not silver-side rejection.

No SCD2. The `commission_modifier` column is the one realistic SCD2 candidate. When underwriting reprices a program (HCP modifier goes from 1.15 to 1.20 because the loss ratio improved), every future commission calculation needs the new value but historical commissions need to reference the rate that was in effect at payout time. Deferred to v2 with the rest of the SCD2 work.

No business logic. The commission_modifier is just stored here; it gets applied in silver_commission_payouts.


In [2]:
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

# Same batch ID convention as silver_carriers.
SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Bronze needs three-part name (non-default lakehouse). Silver is the default
# so silver_programs stays unqualified.
SOURCE_TABLE = "bronze_citadel_mga.dbo.bronze_programs"
TARGET_TABLE = "silver_programs"

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"Reading from:    {SOURCE_TABLE}")
print(f"Writing to:      {TARGET_TABLE}")

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 4, Finished, Available, Finished, False)

Silver batch ID: silver_20260601_211535_51108e25
Reading from:    bronze_citadel_mga.dbo.bronze_programs
Writing to:      silver_programs


In [3]:
# Read bronze_programs and confirm what we're working with before promoting.
bronze_df = spark.read.table(SOURCE_TABLE)

print(f"Row count: {bronze_df.count()}")
print(f"Columns:   {bronze_df.columns}")
print()
bronze_df.printSchema()
print()
bronze_df.show(truncate=False)

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 5, Finished, Available, Finished, False)

Row count: 10
Columns:   ['program_code', 'program_name', 'base_premium_min', 'base_premium_max', 'loss_ratio_target', 'commission_modifier', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

root
 |-- program_code: string (nullable = true)
 |-- program_name: string (nullable = true)
 |-- base_premium_min: integer (nullable = true)
 |-- base_premium_max: integer (nullable = true)
 |-- loss_ratio_target: double (nullable = true)
 |-- commission_modifier: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _ingestion_method: string (nullable = true)


+------------+----------------------+----------------+----------------+-----------------+-------------------+--------------------------+------------+-------------------------------+-----------------+
|program_code|program_name          |base_premium_min|base_premium_max|loss_ratio_target|commission_m

In [4]:
# Promote bronze to silver. Types are clean from bronze so no casting needed.
# Drop bronze lineage we don't carry forward, rename _batch_id to avoid
# collision with the silver batch ID we add below.
silver_df = (
    bronze_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

print(f"Row count: {silver_df.count()}")
print(f"Columns:   {silver_df.columns}")
print()
silver_df.show(truncate=False)

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 6, Finished, Available, Finished, False)

Row count: 10
Columns:   ['program_code', 'program_name', 'base_premium_min', 'base_premium_max', 'loss_ratio_target', 'commission_modifier', '_bronze_batch_id', '_silver_load_timestamp', '_silver_batch_id']

+------------+----------------------+----------------+----------------+-----------------+-------------------+-------------------------------+--------------------------+-------------------------------+
|program_code|program_name          |base_premium_min|base_premium_max|loss_ratio_target|commission_modifier|_bronze_batch_id               |_silver_load_timestamp    |_silver_batch_id               |
+------------+----------------------+----------------+----------------+-----------------+-------------------+-------------------------------+--------------------------+-------------------------------+
|DIET        |Dietary Supplements   |2000            |50000           |0.35             |1.05               |bronze_20260601_210535_0c1a3e36|2026-06-01 21:17:30.957119|silver_20260601_2115

In [5]:
# Pre-write validation.
null_program_code = silver_df.filter(col("program_code").isNull()).count()
null_modifier = silver_df.filter(col("commission_modifier").isNull()).count()
total_count = silver_df.count()
distinct_code_count = silver_df.select("program_code").distinct().count()
bronze_count = bronze_df.count()

print(f"Total rows:           {total_count}")
print(f"Bronze row count:     {bronze_count}")
print(f"Null program_code:    {null_program_code}")
print(f"Null commission_modifier: {null_modifier}")
print(f"Distinct program_code:    {distinct_code_count}")
print()

assert null_program_code == 0, f"Found {null_program_code} rows with null program_code"
assert null_modifier == 0, f"Found {null_modifier} rows with null commission_modifier"
assert distinct_code_count == total_count, (
    f"Duplicate program_code found: {total_count} rows, {distinct_code_count} distinct"
)
assert total_count == bronze_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={total_count}"
)

print("All checks passed.")

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 7, Finished, Available, Finished, False)

Total rows:           10
Bronze row count:     10
Null program_code:    0
Null commission_modifier: 0
Distinct program_code:    10

All checks passed.


In [6]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 8, Finished, Available, Finished, False)

Wrote 10 rows to silver_programs


In [7]:
verify_df = spark.read.table(TARGET_TABLE)

row_count = verify_df.count()
col_count = len(verify_df.columns)

required_lineage = ["_bronze_batch_id", "_silver_load_timestamp", "_silver_batch_id"]
has_lineage = all(c in verify_df.columns for c in required_lineage)

print(f"silver_programs verified:")
print(f"  Rows:         {row_count}")
print(f"  Columns:      {col_count}")
print(f"  Lineage cols: {'present' if has_lineage else 'MISSING'}")
print()
print("Sample:")
verify_df.select(
    "program_code", "program_name", "commission_modifier",
    "_bronze_batch_id", "_silver_batch_id"
).show(truncate=False)

StatementMeta(, 7207a738-53c9-4ef3-828f-dc9b7738880e, 9, Finished, Available, Finished, False)

silver_programs verified:
  Rows:         10
  Columns:      9
  Lineage cols: present

Sample:
+------------+----------------------+-------------------+-------------------------------+-------------------------------+
|program_code|program_name          |commission_modifier|_bronze_batch_id               |_silver_batch_id               |
+------------+----------------------+-------------------+-------------------------------+-------------------------------+
|DIET        |Dietary Supplements   |1.05               |bronze_20260601_210535_0c1a3e36|silver_20260601_211535_51108e25|
|ENV         |Environmental         |1.05               |bronze_20260601_210535_0c1a3e36|silver_20260601_211535_51108e25|
|HCP         |Healthcare Providers  |1.15               |bronze_20260601_210535_0c1a3e36|silver_20260601_211535_51108e25|
|ENRG        |Energy                |1.1                |bronze_20260601_210535_0c1a3e36|silver_20260601_211535_51108e25|
|LIFE        |Life Sciences         |1.1          